<a href="https://colab.research.google.com/github/Dacon-TeamPK/dacon-physics-vision-ai/blob/model/resnet/dacon_%EA%B5%AC%EC%A1%B0%EB%AC%BC_%EC%95%88%EC%A0%95%EC%84%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#MultiResNet(수정 있음)



In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

In [4]:
os.chdir('/content/drive/MyDrive/open_dacon')

print(os.getcwd())

/content/drive/MyDrive/open_dacon


In [5]:
import os
import random
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms
from torchvision.models import ResNet18_Weights, ResNet50_Weights

from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm

# 하이퍼파라미터 설정
CFG = {
    'IMG_SIZE': 224,
    'EPOCHS': 10,
    'LEARNING_RATE': 1e-4,   # resnet18 기본 추천
    'BATCH_SIZE': 32,
    'SEED': 42,
    'MODEL_NAME': 'resnet18',   # 'resnet18' 또는 'resnet50'
    'DROPOUT': 0.3
}

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG['SEED'])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [6]:
def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG['SEED'])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
# 1. 데이터 로드
train_df = pd.read_csv('./train.csv')
val_df = pd.read_csv('./dev.csv')

print(f"학습 데이터 개수: {len(train_df)}")
print(f"검증 데이터 개수: {len(val_df)}")

학습 데이터 개수: 1000
검증 데이터 개수: 100


In [8]:
class MultiViewDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {'stable': 0, 'unstable': 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        sample_id = str(self.df.iloc[idx]['id'])
        # 폴더 경로 설정
        folder_path = os.path.join(self.root_dir, sample_id)

        # 2개 뷰 로드
        views = []
        for name in ["front", "top"]:
            img_path = os.path.join(folder_path, f"{name}.png")
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            views.append(image)

        # 테스트(추론) 모드일 경우 이미지 리스트만 반환
        if self.is_test:
            return views

        # 학습/검증 모드일 경우 라벨 함께 반환
        label = self.label_map[self.df.iloc[idx]['label']]
        return views, label

In [9]:
train_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = MultiViewDataset(train_df, './train', train_transform, is_test=False)
val_dataset = MultiViewDataset(val_df, './dev', test_transform, is_test=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_df = pd.read_csv('./sample_submission.csv')
test_dataset = MultiViewDataset(test_df, './test', test_transform, is_test=True)
test_loader = DataLoader(
    test_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [10]:
class MultiViewResNet(nn.Module):
    def __init__(self, model_name='resnet18', num_classes=1, dropout=0.3):
        super(MultiViewResNet, self).__init__()

        if model_name == 'resnet18':
            self.backbone = models.resnet18(weights=ResNet18_Weights.DEFAULT)
            feat_dim = 512
        elif model_name == 'resnet50':
            self.backbone = models.resnet50(weights=ResNet50_Weights.DEFAULT)
            feat_dim = 2048
        else:
            raise ValueError("model_name must be 'resnet18' or 'resnet50'")

        self.feature_extractor = nn.Sequential(*list(self.backbone.children())[:-1])

        # f1, f2, |f1-f2|, f1*f2
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim * 4, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, num_classes)
        )

    def forward(self, views):
        f1 = self.feature_extractor(views[0]).view(views[0].size(0), -1)
        f2 = self.feature_extractor(views[1]).view(views[1].size(0), -1)

        diff = torch.abs(f1 - f2)
        prod = f1 * f2

        combined = torch.cat([f1, f2, diff, prod], dim=1)
        return self.classifier(combined)

In [11]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    train_loss = 0.0

    for views, labels in tqdm(loader, desc="Training"):
        views = [v.to(device) for v in views]
        labels = labels.to(device).float()

        optimizer.zero_grad()
        outputs = model(views).view(-1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    return train_loss / len(loader)


def validate(model, loader, criterion, device):
    model.eval()
    val_loss = 0.0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for views, labels in tqdm(loader, desc="Validation"):
            views = [v.to(device) for v in views]
            labels = labels.to(device).float()

            outputs = model(views).view(-1)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            probs = torch.sigmoid(outputs)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_probs = np.array(all_probs, dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.float64)

    eps = 1e-15
    p = np.clip(all_probs, eps, 1 - eps)
    logloss_score = -np.mean(all_labels * np.log(p) + (1 - all_labels) * np.log(1 - p))
    acc_score = np.mean((all_probs > 0.5) == all_labels)

    return val_loss / len(loader), logloss_score, acc_score

In [12]:
model = MultiViewResNet(
    model_name=CFG['MODEL_NAME'],
    dropout=CFG['DROPOUT']
).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=CFG['LEARNING_RATE'],
    weight_decay=1e-4
)

scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2
)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 194MB/s]


In [13]:
best_logloss = float('inf')

for epoch in range(1, CFG['EPOCHS'] + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_logloss, val_acc = validate(model, val_loader, criterion, device)

    print(f"\nEpoch [{epoch}/{CFG['EPOCHS']}]")
    print(f"  - Train Loss   : {train_loss:.4f}")
    print(f"  - Val Loss     : {val_loss:.4f}")
    print(f"  - Val LogLoss  : {val_logloss:.6f}")
    print(f"  - Val Acc      : {val_acc:.4f}")
    print(f"  - LR           : {optimizer.param_groups[0]['lr']:.8f}")

    scheduler.step(val_logloss)

    if val_logloss < best_logloss:
        best_logloss = val_logloss
        torch.save(model.state_dict(), "best_model.pth")
        print("  -> Best model saved!")

Training:   0%|          | 0/32 [00:00<?, ?it/s]

Validation:   0%|          | 0/4 [00:00<?, ?it/s]


Epoch [1/10]
  - Train Loss   : 0.4939
  - Val Loss     : 0.7889
  - Val LogLoss  : 0.922183
  - Val Acc      : 0.5300
  - LR           : 0.00010000
  -> Best model saved!


Training:   0%|          | 0/32 [00:00<?, ?it/s]

Validation:   0%|          | 0/4 [00:00<?, ?it/s]

...


Epoch [9/10]
  - Train Loss   : 0.0474
  - Val Loss     : 0.6496
  - Val LogLoss  : 0.748391
  - Val Acc      : 0.7000
  - LR           : 0.00005000


Training:   0%|          | 0/32 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79b5a13b74c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79b5a13b74c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Validation:   0%|          | 0/4 [00:00<?, ?it/s]


Epoch [10/10]
  - Train Loss   : 0.0438
  - Val Loss     : 0.5967
  - Val LogLoss  : 0.686202
  - Val Acc      : 0.7200
  - LR           : 0.00002500


In [14]:
model = MultiViewResNet(
    model_name=CFG['MODEL_NAME'],
    dropout=CFG['DROPOUT']
).to(device)

# best model 불러오기
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()

all_probs = []

with torch.no_grad():
    for views in tqdm(test_loader, desc="Inference"):
        views = [v.to(device) for v in views]

        outputs = model(views).view(-1)
        probs = torch.sigmoid(outputs)

        all_probs.extend(probs.cpu().numpy())

all_probs = np.array(all_probs)

submission = pd.DataFrame({
    'id': test_df['id'],
    'unstable_prob': all_probs,
    'stable_prob': 1.0 - all_probs
})

submission.to_csv('submission.csv', index=False)

print("submission.csv 생성 완료!")

Inference:   0%|          | 0/32 [00:00<?, ?it/s]

submission.csv 생성 완료!
